# Laya × Memory Fusion V3-Turbo — Fast Native Bidirectional Fit

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vtavakkoli/TinyCeNN-LM/blob/main/notebooks/Laya_MemoryFusion_Colab.ipynb)

This version keeps **`convaiinnovations/laya` unchanged as the teacher** and targets **ModernBERT full-attention layer 12** first.

### V3-Turbo convergence changes
- Native bidirectional symmetric local Q/K/V + full-sequence Hedgehog + forward/backward GDN2.
- Adds a nearly free **direct-V branch** so the fusion starts from a strong self-token anchor.
- **Three learning-rate groups:** fast router/gains, slower feature maps/memory, conservative `Wo`.
- LR warm-up protects against the large step-1 gradient spike seen in the previous run.
- Adaptive loss: cosine weight rises from `0.30 → 0.85` once NMSE enters the near-gate region.
- GDN2 activates earlier at roughly `NMSE ≤ 0.36 / cosine ≥ 0.80`.
- Cached training batches are reshuffled every pass instead of replayed in one fixed order.
- Teacher I/O cache is kept on GPU when possible, removing CPU→GPU copies during fitting.
- Only **160 steps per round** (maximum two rounds), with best-probe restore and plateau LR cuts.
- Strict acceptance remains unchanged: `NMSE ≤ 0.20`, `cosine ≥ 0.90`, agreement/KL/accuracy gates unchanged.


## 1. Setup
A T4/L4/A100 runtime is recommended. The setup pulls the latest TinyCeNN-LM and Laya source, then performs a syntax preflight before training.


In [ ]:
import os, sys, subprocess, pathlib, importlib, compileall
os.environ["USE_TF"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

if pathlib.Path("/content").exists():
    WORK = pathlib.Path("/content")
elif pathlib.Path("/kaggle/working").exists():
    WORK = pathlib.Path("/kaggle/working")
else:
    WORK = pathlib.Path.cwd()

REPO = WORK / "TinyCeNN-LM"
if not (REPO / ".git").exists():
    subprocess.check_call(["git", "clone", "-q", "https://github.com/vtavakkoli/TinyCeNN-LM.git", str(REPO)])
else:
    subprocess.check_call(["git", "-C", str(REPO), "pull", "--ff-only", "-q"])

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO)])
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "git+https://github.com/NandhaKishorM/laya.git",
    "datasets", "pandas", "pyarrow", "safetensors"
])

SRC = str(REPO / "src")
if SRC not in sys.path:
    sys.path.insert(0, SRC)
importlib.invalidate_caches()

LAB_SRC = REPO / "src" / "tinycenn_lm" / "laya_lab"
assert compileall.compile_dir(str(LAB_SRC), quiet=1), "Python syntax preflight failed in tinycenn_lm/laya_lab"

import torch, json, pandas as pd
from tinycenn_lm.laya_lab.memory_fusion_v3 import (
    LayaMemoryFusionV3Config,
    run_memory_fusion_v3,
)

print("repo:", REPO)
print("torch:", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")


## 2. Fast layer-12 configuration

V3 deliberately solves **one layer well first**. Layer 12 is full attention and gave the best behavior in the previous Laya experiments. The strict quality gates remain unchanged.

The first phase builds a teacher cache, then performs block-only optimization. This removes the dominant V2 overhead while using a larger and more stable validation probe.


In [ ]:
MODEL_ID = "convaiinnovations/laya"

cfg = LayaMemoryFusionV3Config(
    model_id=MODEL_ID,
    seed=2026,
    output_dir="/content/laya_tinycenn",

    candidate_layer=12,

    feature_dim=64,
    memory_rank=64,
    dilations=(1, 2, 4, 8, 16, 32, 64),

    train_cases=480,
    train_max_len=512,
    batch_size=4,
    train_cache_batches=48,   # 192 cached training examples
    probe_cache_batches=8,    # fixed 32-example probe
    cache_on_device=True,      # keep cached layer I/O on T4/L4/A100 when possible

    # Turbo schedule: fewer steps, more useful movement per step.
    functional_steps=160,
    max_rounds=2,
    check_every=20,
    min_steps_before_check=40,
    fast_learning_rate=1.0e-3,   # fusion/router/gains
    core_learning_rate=5.0e-4,   # Hedgehog + GDN2 feature maps
    output_learning_rate=5.0e-5, # copied Laya Wo
    warmup_steps=10,
    round_lr_decay=0.70,
    weight_decay=2e-4,

    # Push direction/cosine much harder once reconstruction is reasonable.
    cosine_weight=0.30,
    near_gate_cosine_weight=0.85,
    near_gate_nmse=0.38,

    # Start memory refinement earlier than the previous run.
    memory_enable_nmse=0.36,
    memory_enable_cosine=0.80,

    # Strict final acceptance — unchanged.
    max_local_nmse=0.20,
    min_local_cosine=0.90,
    min_teacher_agreement=0.95,
    max_mean_kl=0.05,
    max_accuracy_drop=0.02,

    gate_cases=80,
    final_cases=160,

    decision_refine_steps=30,
    decision_refine_lr_scale=0.25,
)
print(cfg)


## 3. V3-Turbo fitting → strict Laya gate

The trainer caches untouched teacher layer-12 I/O once. The fast phase learns local/Hedgehog/direct-V fusion with aggressive router learning rates. Once the fixed probe enters the near-gate region, cosine receives more weight and forward/backward GDN2 is enabled. Full Laya decision evaluation still runs only after the strict local gate passes.


In [ ]:
teacher, student, report = run_memory_fusion_v3(cfg)


## 4. Result summary
If no layer passes, the final student is restored to original Laya attention and the report explicitly says 'failed_no_accepted_layers'. Identical teacher/student metrics are therefore never presented as a successful conversion.


In [ ]:
summary = pd.DataFrame([
    {
        "model": "Laya teacher",
        **{k: report["teacher_final"].get(k) for k in [
            "accuracy", "soft_accuracy", "brier", "brier_vs_soft",
            "ece", "score_mae", "ms_per_case"
        ]},
    },
    {
        "model": report["architecture"],
        **{k: report["student_final"].get(k) for k in [
            "accuracy", "soft_accuracy", "brier", "brier_vs_soft",
            "ece", "score_mae", "ms_per_case"
        ]},
        "teacher_agreement": report["student_final"].get("teacher_agreement"),
        "teacher_KL": report["student_final"].get("mean_teacher_kl"),
    },
])
display(summary)

print("Status:", report["status"])
print("Accepted attention layers:", report["accepted_layers"])
print("Replacement trainable parameters:", f'{report["replacement_trainable_parameters"]:,}')
print("Gate/final disjoint:", report["gate_final_disjoint"])
print("Latency:", json.dumps(report["latency"], indent=2))


## 5. Round-by-round diagnostics

The important values are the fixed **PROBE** lines. Compared with the previous run, V3-Turbo should reach roughly `NMSE < 0.32 / cosine > 0.82` noticeably earlier than step 160. When NMSE falls below `0.38`, the log will show `cw=0.85`; when memory turns on, it prints the GDN2 refinement message.


In [ ]:
rows = []
for h in report["history"]:
    gate = h.get("gate") or {}
    local = h.get("local") or {}
    rows.append({
        "layer": h.get("layer"),
        "round": h.get("round"),
        "stage": h.get("stage"),
        "local_pass": h.get("local_pass"),
        "accepted": h.get("accepted"),
        "best_step": local.get("step"),
        "nmse": local.get("nmse"),
        "cosine": local.get("cosine"),
        "teacher_agreement": gate.get("teacher_agreement"),
        "mean_teacher_kl": gate.get("mean_teacher_kl"),
        "accuracy": gate.get("accuracy"),
        "accuracy_drop": h.get("accuracy_drop"),
    })
display(pd.DataFrame(rows))


## 6. Laya Router smoke test

The adapted Agent keeps Laya's public API. The example below attaches the already-loaded student without loading a second model.


In [ ]:
from laya import Router

state = {
    "from": "user@acme.com",
    "subject": "Duplicate charge on invoice #4411",
    "body": "Hi, we were billed twice for March. Please refund the duplicate today or we will cancel our plan.",
}
questions = {
    "department": {
        "type": "choice",
        "instructions": "Which department should handle this request?",
        "criteria": {
            "billing": "invoices, payments, refunds",
            "technical": "bugs, outages, system errors",
            "sales": "pricing, new contracts",
            "other": "everything else",
        },
    },
    "urgency": {
        "type": "score",
        "instructions": "How urgent is this request?",
        "criteria": ["not urgent", "soon", "critical deadline or blocking issue"],
    },
    "churn_risk": {
        "type": "noul",
        "instructions": "Does the user threaten to cancel or leave?",
    },
    "refund_requested": {
        "type": "noul",
        "instructions": "Does the user explicitly request a refund?",
    },
}

router = Router(preload=False)
router.attach("english", student)
res = router.predict(state, questions, model="english")

print("Department       :", res["answers"]["department"]["choice"])
print("Urgency score    :", res["answers"]["urgency"]["score"])
print("Churn risk       :", res["answers"]["churn_risk"]["noul"])
print("Refund requested :", res["answers"]["refund_requested"]["noul"])
print("Routing          :", res["routing"]["model"])
print("\nTeacher/student raw comparison:")
print(json.dumps(report["demo"], indent=2, ensure_ascii=False))


## 7. Saved outputs

V3 writes its adapter, report, and best checkpoint for each functional round under `/content/laya_tinycenn/memory_fusion_v3/`.


In [ ]:
from pathlib import Path
out = Path(cfg.output_dir) / "memory_fusion_v3"
print("Adapter:", out / "adapter.pt")
print("Report :", out / "report.json")
print("Round checkpoints:")
for p in sorted(out.glob("layer_*_round_*.pt")):
    print(" -", p.name)
print("\nReport preview:")
print((out / "report.json").read_text()[:5000])
